<a href="https://colab.research.google.com/github/keertiam8/gnn-congestion/blob/main/gnn_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
# cell A
import shutil, os
shutil.rmtree("/content/closed-loop-placement", ignore_errors=True)
os.chdir("/content")
!git clone https://github.com/keertiam8/gnn-congestion closed-loop-placement
os.chdir("/content/closed-loop-placement")
!git log --oneline -3

Cloning into 'closed-loop-placement'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 121 (delta 63), reused 76 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 870.40 KiB | 3.22 MiB/s, done.
Resolving deltas: 100% (63/63), done.
13c9dce (HEAD -> main, origin/main, origin/hrdya, origin/HEAD) Created using Colab
b2931e2 Add eval_gnn.py: real NRMSE/SSIM comparison against GPDL baseline
e828b4e Add explicit variance-matching term, lower default peak_weight


In [2]:
# cell B
!pip install --quiet gdown scipy scikit-image


In [3]:
pip install --quiet torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.7 MB/s eta 0:00:00


In [4]:
!python scripts/colab_prepare_congestion_data.py --num-samples 500 --no-eval



=== macro_region ===
downloading...
Downloading...
From: https://drive.google.com/uc?id=14n9khpSK56NUZrGUNPPRYmZkAbCOstKq
To: /content/circuitnet_downloads/macro_region.tar.gz
100% 6.10M/6.10M [00:00<00:00, 33.3MB/s]
picked 500 sample IDs from macro_region
/content/closed-loop-placement/scripts/colab_prepare_congestion_data.py:173: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extract(member, extract_dir)
extracted 500 files from macro_region
deleted /content/circuitnet_downloads/macro_region.tar.gz to free space

=== rudy ===
downloading...
Downloading...
From (original): https://drive.google.com/uc?id=1KUocSofLvyAFKXu8AXt4j3TPJsiaCjS6
From (redirected): https://drive.google.com/uc?id=1KUocSofLvyAFKXu8AXt4j3TPJsiaCjS6&confirm=t&uuid=ca02fcf1-443a-4d4d-aa60-6d1ebea1f28f
To: /content/circuitnet_downloads/rudy.tar.gz
100% 2.73G/2.73G [00:40<00:00, 66.9MB/s

In [5]:
!python scripts/preprocess_circuitnet.py \
    --root data/circuitnet_raw/congestion \
    --out /content/circuitnet_graphs \
    --limit 500


Done. 500 graphs written to /content/circuitnet_graphs, 0 skipped.


In [ ]:
import importlib, gnn.model
importlib.reload(gnn.model)
from gnn.model import CongestionGNN


ckpt = torch.load("checkpoints/pretrained.pt", map_location="cpu")
model = CongestionGNN(in_channels=5)
model.load_state_dict(ckpt["model"] if "model" in ckpt else ckpt)
model.eval()

g = torch.load(glob.glob("/content/circuitnet_graphs/*.pt")[0], weights_only=False)
with torch.no_grad():
    pred = model(g.x, g.edge_index)
print("pred std:", pred.std().item(), "label std:", g.y.std().item())
print("pred mean:", pred.mean().item(), "label mean:", g.y.mean().item())


pred std: 0.01132422499358654 label std: 0.018049227073788643
pred mean: 0.1289944052696228 label mean: 0.14110025763511658


In [19]:
!python scripts/train_pretrain.py \
    --data /content/circuitnet_graphs \
    --epochs 50 \
    --out checkpoints/pretrained.pt \
    --lr 3e-4 \
    --batch-size 4

Loaded 500 graphs
in_channels=5, device=cuda
epoch 001  train_loss=0.0078  val_loss=0.0046
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 002  train_loss=0.0046  val_loss=0.0040
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 003  train_loss=0.0042  val_loss=0.0040
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 004  train_loss=0.0041  val_loss=0.0038
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 005  train_loss=0.0040  val_loss=0.0039
epoch 006  train_loss=0.0039  val_loss=0.0038
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 007  train_loss=0.0040  val_loss=0.0039
epoch 008  train_loss=0.0039  val_loss=0.0037
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 009  train_loss=0.0038  val_loss=0.0037
epoch 010  train_loss=0.0037  val_loss=0.0041
epoch 011  train_loss=0.0037  val_loss=0.0036
  -> saved best checkpoint to checkpoints/pretrained.pt
epoch 012  train_loss=0.0036  val_loss=0.0051
epoch 013  

In [21]:
!git config --global user.email "keertiam8@gmail.com"
!git config --global user.name "Keerti A M"


In [23]:
!git fetch origin main
!git checkout main


remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 5 (delta 3), reused 5 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 857 bytes | 857.00 KiB/s, done.
From https://github.com/keertiam8/gnn-congestion
 * branch            main       -> FETCH_HEAD
   13c9dce..33e9915  main       -> origin/main
Already on 'main'
Your branch is behind 'origin/main' by 2 commits, and can be fast-forwarded.
  (use "git pull" to update your local branch)


In [24]:
!git pull


Updating 13c9dce..33e9915
Fast-forward
 .gitignore | 1 -
 1 file changed, 1 deletion(-)


In [26]:
!ls -lh /content/closed-loop-placement/checkpoints/pretrained.pt


-rw-r--r-- 1 root root 508K Jul 28 09:54 /content/closed-loop-placement/checkpoints/pretrained.pt


In [28]:
!git ls-files checkpoints/pretrained.pt


checkpoints/pretrained.pt


In [27]:
!git reset --soft HEAD~1
!git add checkpoints/pretrained.pt
!git commit -m "Pretrained GNN: epoch 23, 500 CircuitNet samples"
!git push



[main 1d17992] Pretrained GNN: epoch 23, 500 CircuitNet samples
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 checkpoints/pretrained.pt
fatal: could not read Username for 'https://github.com': No such device or address
